# 01 · Is the dataset complete, and what needs cleaning?

**In one sentence:** we load the PhiUSIIL dataset, check that the file is complete, make phishing the class we predict, and remove the problems in the URLs that would let a model cheat.

**Why:** every later notebook trains and tests on this data. If the file is cut off, the labels point the wrong way, or the same URL appears twice, every score after this one is wrong, and nothing would warn us.

### Words you need for this notebook
- **Row:** one link (URL) with its label and 54 other columns describing it.
- **Label:** the answer we want to predict: is this link phishing or legitimate?
- **Positive class:** the class that metrics like precision, recall and F1 are calculated for. In scikit-learn it is the class written as `1`.
- **Shortcut (or leak):** a pattern that separates the classes in this dataset only because of how the data was collected, not because of anything about phishing. A model that learns it scores well here and fails on real links.

### What happens below
1. Load the raw file and check it has the published size.
2. Flip the label so that 1 means phishing.
3. Look at trailing slashes, which give the answer away.
4. Look for duplicate URLs and URLs labelled both ways.
5. Apply the cleaning and count what was removed.
6. Check how many phishing and legitimate links are left.
7. Note two more shortcuts for a later milestone.

All the code that does the work lives in `src/phishguard/data.py` (tested in `tests/test_data.py`). This notebook shows the evidence behind each step. The decisions themselves are D1 to D4 in `docs/DECISIONS.md`.

## Step 1 · Load the raw file and check it is complete

PhiUSIIL is published with **235,795 rows and 55 columns**. Our first cleaning notebook only saw 196,403 rows, because the Colab upload had been cut off, and nothing complained.

So `data.load_raw()` now checks the shape of the file and stops with an error if it is wrong, instead of carrying on silently. It also stops if any value is missing or if the label has values other than 0 and 1.

- `pandas` (imported as `pd`) is the standard Python library for tables. A table is called a **DataFrame**.
- `raw.shape` gives (number of rows, number of columns).

In [1]:
import pandas as pd
from phishguard import data

In [2]:
raw = data.load_raw()
raw.shape

(235795, 55)

**How to read this:** `(235795, 55)` is exactly the published size, so the file is complete.

## Step 2 · Make phishing the positive class

In the original data, `label = 1` means **legitimate** and `label = 0` means **phishing**.

scikit-learn calculates precision, recall and F1 for class `1`. We want those numbers to describe how well we catch **phishing**, so phishing must be `1`.

`data.add_target()` creates a new column `is_phishing` (1 = phishing) and deletes `label`, so nobody can use the old encoding by mistake. `data.TARGET` is just the name `"is_phishing"`, so every notebook refers to the label the same way.

`pd.crosstab` counts how often each pair of values occurs: here, old label against new label.

In [3]:
labelled = data.add_target(raw)
pd.crosstab(raw["label"], labelled[data.TARGET], rownames=["label (original)"], colnames=[data.TARGET])

is_phishing,0,1
label (original),,
0,0,100945
1,134850,0


**How to read this:** every row with old `label = 0` now has `is_phishing = 1` (100,945 rows), and every row with old `label = 1` now has `is_phishing = 0` (134,850 rows). The two zeros show that no row was mapped the wrong way.

## Step 3 · Does a trailing slash give the answer away?

Before cleaning anything, we look at the data first. The first thing we check is whether URLs end with `/`.

**The code, piece by piece:**
- `str.endswith("/")` gives True or False for every URL.
- `groupby(...)` splits the rows into groups, here one group per class (0 = legitimate, 1 = phishing).
- `.mean()` then averages True/False inside each group. True counts as 1 and False as 0, so the average is the **share** of URLs ending in `/`.

In [4]:
ends_with_slash = labelled["URL"].str.endswith("/")
ends_with_slash.groupby(labelled[data.TARGET]).mean().rename("share of URLs ending in '/'")

is_phishing
0    0.000000
1    0.364684
Name: share of URLs ending in '/', dtype: float64

**How to read this:** about 36% of phishing URLs (0.3647) end in `/`, and **no** legitimate URL does (0.0).

This is a side effect of how the two classes were collected, not something about phishing itself. A model would learn "ends in / → phishing", score well on the test set, and fail on real links. So the cleaning removes the trailing slash (D3).

## Step 4 · Duplicate URLs and URLs labelled both ways

**The code, piece by piece:**
- `normalised` is each URL with spaces trimmed (`str.strip()`) and trailing slashes removed (`str.rstrip("/")`), which is what the cleaning will do.
- `duplicated()` marks every row whose URL already appeared earlier. `.sum()` counts them.
- `groupby(normalised)[...].nunique()` counts how many **different** labels each URL has. A URL with more than one label is a **conflict**.
- The last line shows the rows of every conflicting URL.

In [5]:
normalised = labelled["URL"].str.strip().str.rstrip("/")
print("exact duplicate URLs:          ", labelled["URL"].duplicated().sum())
print("duplicates after removing '/': ", normalised.duplicated().sum())

labels_per_url = labelled.groupby(normalised)[data.TARGET].nunique()
conflicts = labels_per_url[labels_per_url > 1].index
labelled.loc[normalised.isin(conflicts), ["URL", data.TARGET]]

exact duplicate URLs:           425
duplicates after removing '/':  1590


,URL,is_phishing
36057,https://www.education.gouv.fr/,1
84900,https://www.education.gouv.fr,0


**How to read this:**
- There are **425** exact duplicate URLs. Once the trailing `/` is removed, more URLs become identical, and there are **1,590**.
- Duplicates must go. Otherwise the same URL can end up in both train and test, and test scores look better than they are.
- Exactly one URL has a conflict: `https://www.education.gouv.fr` is labelled phishing with a slash and legitimate without one. That's **label noise**. We can't know which label is right, so we drop every copy (D4).

## Step 5 · Apply the cleaning

`data.clean()` does the three steps above, in this order:
1. remove spaces and the trailing `/` from every URL;
2. drop every row of a URL that has two different labels;
3. drop duplicate URLs, keeping the first copy.

It returns the cleaned table and a small report of how many rows each step removed.

In [6]:
clean, report = data.clean(labelled)
report

{'rows_in': 235795,
 'conflicting_label_rows_dropped': 2,
 'duplicate_rows_dropped': 1589,
 'rows_out': 234204}

**How to read this:** we start with 235,795 rows, drop the **2** rows of the conflicting URL and **1,589** duplicates, and keep **234,204** rows. (1,589 is one fewer than the 1,590 duplicates in Step 4, because the two `education.gouv.fr` rows were one of those duplicates and are already gone.)

## Step 6 · How many phishing and legitimate links are left?

`value_counts()` counts the rows in each class. We rename 0 and 1 to readable names and add each class's share in percent.

In [7]:
counts = clean[data.TARGET].value_counts().rename({0: "legitimate", 1: "phishing"})
pd.DataFrame({"count": counts, "percent": (counts / counts.sum() * 100).round(2)})

,count,percent
is_phishing,,
legitimate,134849,57.58
phishing,99355,42.42


**How to read this:** 134,849 legitimate links (57.58%) and 99,355 phishing links (42.42%).

The classes are only mildly imbalanced (about 58/42). That doesn't need resampling, the trick of copying or deleting rows to make the classes equal. But it does mean two things for every later notebook:
- we report F1 and ROC-AUC next to accuracy, because accuracy alone can hide poor results on the smaller class;
- we use **stratified** splits: each part of the split keeps the same share of phishing (about 42%), so no part is easier by chance.

## Step 7 · Other shortcuts we noticed (for a later milestone)

We don't fix these here, because what to do about them is a modelling decision, not a cleaning one.

The code looks at two of the provided columns:
- `IsHTTPS`: 1 if the link uses `https://`;
- `URLSimilarityIndex`: a score up to 100 for how similar the link is to a list of known legitimate sites.

For each class it calculates the share of rows where the column is at its top value (1 for HTTPS, 100 for similarity). `agg(lambda s: ...)` applies that small calculation to each column within each class.

In [8]:
clean.groupby(data.TARGET)[["IsHTTPS", "URLSimilarityIndex"]].agg(lambda s: (s == s.max()).mean()).rename(
    columns={"IsHTTPS": "share with HTTPS", "URLSimilarityIndex": "share with similarity = 100"})

,share with HTTPS,share with similarity = 100
is_phishing,,
0,1.00000,1.000000
1,0.48842,0.007881


**How to read this:**
- **Every** legitimate link uses HTTPS and has `URLSimilarityIndex = 100` (both 1.0).
- Among phishing links, only about 49% use HTTPS (0.488), and under 1% have similarity 100 (0.0079).
- So either column alone nearly separates the two classes. A model given these columns doesn't have to learn anything about phishing.
- Several other columns describe the downloaded web page (`LineOfCode`, `NoOfImage`, …), not the URL. A user of our app only gives us a link, so those can't be used there.

## What we learned
1. **The file is complete:** 235,795 rows and 55 columns, the published size. The code now checks this on every load, so a cut-off file can't slip through again.
2. **Phishing is now the positive class** (`is_phishing = 1`), so precision, recall and F1 describe how well we catch phishing.
3. **A trailing `/` was a shortcut:** about 36% of phishing URLs had one and no legitimate URL did. We remove it.
4. **After removing 2 conflicting rows and 1,589 duplicates, 234,204 rows remain:** 57.58% legitimate and 42.42% phishing.
5. **The dataset's own columns contain stronger shortcuts:** every legitimate link has HTTPS and a similarity score of 100. Notebooks 06 and 07 (Milestone 3) deal with this.

Every later notebook gets exactly these rows with two lines:
```python
from phishguard.data import load_clean, TARGET
df = load_clean()
```

**Next, notebook 01b:** split the cleaned data into train, validation and test so that no website appears in more than one part.